In [23]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/baostock')
import datasource
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.get_codes()
code = codes[0]
dt = datasource.get_data(code)
dt.head()

,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST
date,,,,,,,,,,,,,
2010-01-04,sh.600000,5.079055,5.088362,4.923170,4.930150,5.046482,66191338,1.419984e+09,2,0.835129,1,-2.3052,0
2010-01-05,sh.600000,4.981336,5.020889,4.837085,4.967376,4.930150,115147943,2.436891e+09,2,1.452808,1,0.7551,0
2010-01-06,sh.600000,4.953417,4.955743,4.858024,4.869658,4.967376,96782575,2.034174e+09,2,1.221095,1,-1.9672,0
2010-01-07,sh.600000,4.858024,4.895251,4.723079,4.760305,4.869658,85236072,1.761801e+09,2,1.075414,1,-2.2456,0
2010-01-08,sh.600000,4.732386,4.839411,4.723079,4.813818,4.760305,65707646,1.349532e+09,2,0.829026,1,1.1241,0


In [24]:
dt['isUp'] = dt['close'] / dt['open'] >= 1.09
dt.head()

,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST,isUp
date,,,,,,,,,,,,,,
2010-01-04,sh.600000,5.079055,5.088362,4.923170,4.930150,5.046482,66191338,1.419984e+09,2,0.835129,1,-2.3052,0,False
2010-01-05,sh.600000,4.981336,5.020889,4.837085,4.967376,4.930150,115147943,2.436891e+09,2,1.452808,1,0.7551,0,False
2010-01-06,sh.600000,4.953417,4.955743,4.858024,4.869658,4.967376,96782575,2.034174e+09,2,1.221095,1,-1.9672,0,False
2010-01-07,sh.600000,4.858024,4.895251,4.723079,4.760305,4.869658,85236072,1.761801e+09,2,1.075414,1,-2.2456,0,False
2010-01-08,sh.600000,4.732386,4.839411,4.723079,4.813818,4.760305,65707646,1.349532e+09,2,0.829026,1,1.1241,0,False


In [32]:
# 是否涨停
dt['isUp'] = dt['close'] / dt['open'] >= MAX_UP_RATE
dt.head()

,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST,isUp,continuous1
date,,,,,,,,,,,,,,,
2010-01-04,sh.600000,5.079055,5.088362,4.923170,4.930150,5.046482,66191338,1.419984e+09,2,0.835129,1,-2.3052,0,False,0.0
2010-01-05,sh.600000,4.981336,5.020889,4.837085,4.967376,4.930150,115147943,2.436891e+09,2,1.452808,1,0.7551,0,False,0.0
2010-01-06,sh.600000,4.953417,4.955743,4.858024,4.869658,4.967376,96782575,2.034174e+09,2,1.221095,1,-1.9672,0,False,0.0
2010-01-07,sh.600000,4.858024,4.895251,4.723079,4.760305,4.869658,85236072,1.761801e+09,2,1.075414,1,-2.2456,0,False,0.0
2010-01-08,sh.600000,4.732386,4.839411,4.723079,4.813818,4.760305,65707646,1.349532e+09,2,0.829026,1,1.1241,0,False,0.0


In [28]:
day=0
dt.loc[dt['isUp'] == 0, 'isUp'] = 0
dt.loc[dt['isUp'] == 1, 'isUp'] = 1
dt[f'continuous{day+1}'] = dt['isUp'].rolling(window=day+1).sum()
dt.head()

C:\Users\xuhen\AppData\Local\Temp\ipykernel_13556\1509204651.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  dt.loc[dt['isUp'] == 0, 'isUp'] = 0


,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST,isUp,continuous1
date,,,,,,,,,,,,,,,
2010-01-04,sh.600000,5.079055,5.088362,4.923170,4.930150,5.046482,66191338,1.419984e+09,2,0.835129,1,-2.3052,0,0,0.0
2010-01-05,sh.600000,4.981336,5.020889,4.837085,4.967376,4.930150,115147943,2.436891e+09,2,1.452808,1,0.7551,0,0,0.0
2010-01-06,sh.600000,4.953417,4.955743,4.858024,4.869658,4.967376,96782575,2.034174e+09,2,1.221095,1,-1.9672,0,0,0.0
2010-01-07,sh.600000,4.858024,4.895251,4.723079,4.760305,4.869658,85236072,1.761801e+09,2,1.075414,1,-2.2456,0,0,0.0
2010-01-08,sh.600000,4.732386,4.839411,4.723079,4.813818,4.760305,65707646,1.349532e+09,2,0.829026,1,1.1241,0,0,0.0


In [33]:
# 我这里还是用遍历吧
max_index = len(dt)
isUp_index = list(dt.columns).index('isUp')
if max_index > 30:
    index = 30
    while index < max_index:
        if dt.iat[index, isUp_index] == True:
            # 寻找到一个，然后看看下边有几个
            i = 0
            while index < max_index and   dt.iat[index, isUp_index] == True:
                i = i + 1
                index = index + 1
            # 这里表示寻找到了。
            print(i)
        # 不管有没有，就要下一个。
        index = index + 1

1
1


In [31]:
list(dt.columns)

['code',
 'open',
 'high',
 'low',
 'close',
 'preclose',
 'volume',
 'amount',
 'adjustflag',
 'turn',
 'tradestatus',
 'pctChg',
 'isST',
 'isUp',
 'continuous1']

In [34]:
# 接下来准备全部的股票
MAX_UP_RATE = 1.09  # 最大比率
MAX_DAYS = 30       # 连续多少天
dict_count = {}     # 这里放连续的天数
for i in tqdm(range(len(codes))):
    code = codes[i]
    # 股票
    dt = datasource.get_data(code)
    # 设置是否涨停
    dt['isUp'] = dt['close'] / dt['open'] >= MAX_UP_RATE
    max_index = len(dt)
    isUp_index = list(dt.columns).index('isUp')
    index = 30 # 绕过股票刚发行的那段时间
    # 循环
    while index < max_index:
        # 判断
        if dt.iat[index, isUp_index] == True:
            # 寻找到一个，然后看看下边有几个
            i = 0
            while index < max_index and  dt.iat[index, isUp_index] == True:
                i = i + 1
                index = index + 1
            # 这里表示寻找到了。
            if i in dict_count:
                dict_count[i] += 1
            else:
                dict_count[i] = 1
        # 不管有没有，就要下一个。
        index = index + 1


for key in dict_count:
    print(f'连续{key}天涨停的个数是{dict_count[key]}')
    

100%|██████████████████████████████████████████████████████████████████████████████| 5493/5493 [09:25<00:00,  9.72it/s]

连续1天涨停的个数是127119
连续2天涨停的个数是5337
连续3天涨停的个数是460
连续4天涨停的个数是40
连续5天涨停的个数是9
连续12天涨停的个数是1
连续6天涨停的个数是1


可以看到，连续2天涨停的都不多见的。